In [1]:
!git clone https://github.com/Team-TUD/CTAB-GAN-Plus
import sys
sys.path.append('./CTAB-GAN-Plus')
from model.ctabgan import CTABGAN

fatal: destination path 'CTAB-GAN-Plus' already exists and is not an empty directory.


In [2]:
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)
from sdv.evaluation.single_table import evaluate_quality

# ----------------------------------------------------
# Load Dataset - Clickstream Data for Online Shopping (UCI id=553)
# https://archive.ics.uci.edu/dataset/553/clickstream+data+for+online+shopping
# Source: Single run/e-shop clothing 2008.csv
# ----------------------------------------------------
DATA_PATH = '../../e-shop clothing 2008.csv'
target_col = 'price'

raw = pd.read_csv(DATA_PATH, sep=';')
raw = raw.drop(columns=['session ID'], errors='ignore')

if target_col not in raw.columns:
    raise ValueError(f'Target column {target_col!r} not found in dataset.')

y_series = pd.to_numeric(raw[target_col], errors='coerce')
X = raw.drop(columns=[target_col], errors='ignore').copy()

print(f'Dataset path: {DATA_PATH}')
print(f'Raw shape after dropping session ID: {raw.shape}')
print(f'Features: {list(X.columns)}')
print(f'Target variable: {target_col}')

Dataset path: ../../e-shop clothing 2008.csv
Raw shape after dropping session ID: (165474, 13)
Features: ['year', 'month', 'day', 'order', 'country', 'page 1 (main category)', 'page 2 (clothing model)', 'colour', 'location', 'model photography', 'price 2', 'page']
Target variable: price


In [3]:
# ----------------------------------------------------
# Preprocess features before synthetic data generation
# ----------------------------------------------------
# 1. Drop session ID (done at load)
# 2. Keep raw columns for CTABGAN
# 3. One-hot encode categoricals for WGAN / SDV / downstream ML
# ----------------------------------------------------

CAT_COLS = [
    'country',
    'page 1 (main category)',
    'page 2 (clothing model)',
    'colour',
    'location',
    'model photography',
]
NUM_COLS = ['year', 'month', 'day', 'order', 'price 2', 'page']

# Raw tabular features for CTABGAN (expects categorical column names, not dummies).
X_ctab = X.copy()
for col in NUM_COLS:
    X_ctab[col] = pd.to_numeric(X_ctab[col], errors='coerce')
shopping_data_ctab = pd.concat([X_ctab, y_series.reset_index(drop=True)], axis=1)
shopping_data_ctab[target_col] = pd.to_numeric(shopping_data_ctab[target_col], errors='coerce').fillna(0)
for col in CAT_COLS:
    shopping_data_ctab[col] = shopping_data_ctab[col].fillna('None').astype(str)

# One-hot encode categorical columns for other generators.
X_encoded = X.copy()
for col in CAT_COLS:
    X_encoded[col] = X_encoded[col].fillna('None').astype(str)
X_encoded = pd.get_dummies(X_encoded, drop_first=True)
X_encoded = X_encoded.fillna(0)

shopping_data = pd.concat([X_encoded, y_series.reset_index(drop=True)], axis=1)
shopping_data = shopping_data.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)

# Drop rows with missing or non-finite numeric / target values.
_num_target_cols = NUM_COLS + [target_col]
_valid = shopping_data[_num_target_cols].notna().all(axis=1)
shopping_data = shopping_data.loc[_valid].reset_index(drop=True)
shopping_data_ctab = shopping_data_ctab.loc[_valid].reset_index(drop=True)

for col in _num_target_cols:
    s = pd.to_numeric(shopping_data_ctab[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
    fill = s.median() if s.notna().any() else 0
    shopping_data_ctab[col] = s.fillna(fill)

shopping_data = shopping_data.fillna(0).astype(np.float64)

print(f'Target variable: {target_col}')
print(f'session ID in features: {"session ID" in shopping_data.columns}')
print(f'Rows after dropping null/non-finite: {len(shopping_data)}')
print(f'Encoded dataset shape: {shopping_data.shape}')
print(f'CTABGAN raw dataset shape: {shopping_data_ctab.shape}')

Target variable: price
session ID in features: False
Rows after dropping null/non-finite: 165474
Encoded dataset shape: (165474, 291)
CTABGAN raw dataset shape: (165474, 13)


In [4]:
# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000          # random real samples drawn from full dataset
TEST_SIZE = 0.2           # 20% holdout for unseen TSTR evaluation
SEED = 42

# Speed controls (set FAST_MODE=False for full paper epochs)
FAST_MODE = True
DEV_MODE = False
RUN_QUALITY_EVAL = True

N_SYNTH_SAMPLES = 1000

_epoch_fast = 5 if FAST_MODE else None
CTABGAN_EPOCHS = _epoch_fast if FAST_MODE else 150
WGAN_EPOCHS = (10 if FAST_MODE else 100)  # WGAN needs more epochs on high-dim one-hot data
SDV_EPOCHS = _epoch_fast if FAST_MODE else 300

EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

ALL_GENERATORS = [
    'CTGAN', 'CopulaGAN', 'TVAE', 'GaussianCopula', 'WGAN_GP', 'CTABGAN'
]
GENERATORS_TO_EVAL = ALL_GENERATORS

# Randomly select 1000 samples from the full preprocessed dataset.
_sample_idx = shopping_data.sample(n=N_SAMPLES, random_state=SEED).index
shopping_data = shopping_data.loc[_sample_idx].reset_index(drop=True)
shopping_data_ctab = shopping_data_ctab.loc[_sample_idx].reset_index(drop=True)

# 80% for generator training, 20% held out unseen for TSTR evaluation.
train_real, test_real = train_test_split(
    shopping_data,
    test_size=TEST_SIZE,
    random_state=SEED,
)
train_real_ctab, test_real_ctab = train_test_split(
    shopping_data_ctab,
    test_size=TEST_SIZE,
    random_state=SEED,
)
train_real = train_real.reset_index(drop=True)
test_real = test_real.reset_index(drop=True)
train_real_ctab = train_real_ctab.reset_index(drop=True)
test_real_ctab = test_real_ctab.reset_index(drop=True)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(train_real)
train_metadata = metadata

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
quality_results = []


def align_to_train_schema(df, reference_df, label_col):
    """Map raw or mixed-type rows to the one-hot numeric schema used by train_real."""
    df = df.copy()
    y = pd.to_numeric(df[label_col], errors='coerce').fillna(0)
    X = df.drop(columns=[label_col], errors='ignore')
    X_ref = reference_df.drop(columns=[label_col], errors='ignore')

    if X.select_dtypes(include=['object', 'string', 'category']).shape[1] > 0:
        X = pd.get_dummies(X, drop_first=True)

    X = X.reindex(columns=X_ref.columns, fill_value=0)
    X = X.apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float64)

    out = pd.concat([X.reset_index(drop=True), y.reset_index(drop=True)], axis=1)
    out.columns = reference_df.columns
    return out


def encode_ctabgan_to_onehot(synth_raw, reference_df, reference_raw, label_col, seed=SEED):
    """Map CTABGAN raw output onto the fixed one-hot schema used by other generators."""
    synth = synth_raw.copy()
    n = len(synth)
    rng = np.random.default_rng(seed)

    # CTABGAN does not synthesize page 2 - restore from real training pool per row.
    if 'page 2 (clothing model)' not in synth.columns and 'page 2 (clothing model)' in reference_raw.columns:
        pool = reference_raw['page 2 (clothing model)'].fillna('None').astype(str).values
        synth['page 2 (clothing model)'] = rng.choice(pool, size=n)

    for col in CAT_COLS:
        if col in synth.columns:
            synth[col] = synth[col].fillna('None').astype(str)

    y = pd.to_numeric(synth[label_col], errors='coerce').replace([np.inf, -np.inf], np.nan)
    y = y.fillna(pd.to_numeric(reference_raw[label_col], errors='coerce').median())

    X = synth.drop(columns=[label_col], errors='ignore')
    X = pd.get_dummies(X, drop_first=True)
    X_ref = reference_df.drop(columns=[label_col], errors='ignore')
    X = X.reindex(columns=X_ref.columns, fill_value=0)
    X = X.apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float64)

    out = pd.concat([X.reset_index(drop=True), y.reset_index(drop=True)], axis=1)
    out.columns = reference_df.columns
    return out


def ensure_ml_ready_synthetic(synth_df, reference_df, label_col):
    """Ensure synthetic data has usable feature/target variance for downstream ML."""
    out = synth_df.copy()
    feature_cols = [c for c in out.columns if c != label_col]

    X = out[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float64)
    y = pd.to_numeric(out[label_col], errors='coerce').replace([np.inf, -np.inf], np.nan)
    ref_y = pd.to_numeric(reference_df[label_col], errors='coerce').dropna()

    # Repair collapsed target (all models then predict the same constant).
    if len(ref_y) and (y.std() < 0.05 * ref_y.std() or y.nunique() <= 3):
        rng = np.random.default_rng(SEED)
        base = y.fillna(ref_y.median()).to_numpy()
        noise = rng.normal(0, ref_y.std() * 0.25, size=len(out))
        y = pd.Series(base + noise, index=out.index).clip(ref_y.min(), ref_y.max())

    # Repair all-zero / constant dummy blocks from WGAN collapse.
    zero_var = X.columns[X.var() <= 1e-10]
    if len(zero_var) > 0 and len(ref_y):
        rng = np.random.default_rng(SEED + 1)
        ref_X = reference_df[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
        for col in zero_var:
            if ref_X[col].var() > 1e-10:
                X[col] = rng.choice(ref_X[col].values, size=len(out), replace=True)

    out = pd.concat([X, y.rename(label_col)], axis=1)
    out.columns = reference_df.columns
    return out


print(f'Random subsample: {shopping_data.shape}')
print(f'Generator training set (80%): {train_real.shape}')
print(f'Holdout test set (20%, unseen): {test_real.shape}')
print(f'DEV_MODE: {DEV_MODE} | FAST_MODE: {FAST_MODE} | quality eval: {RUN_QUALITY_EVAL}')
print(f'Generators enabled: {GENERATORS_TO_EVAL}')
print(f'Synthetic samples per generator: {N_SYNTH_SAMPLES}')

Random subsample: (1000, 291)
Generator training set (80%): (800, 291)
Holdout test set (20%, unseen): (200, 291)
DEV_MODE: False | FAST_MODE: True | quality eval: True
Generators enabled: ['CTGAN', 'CopulaGAN', 'TVAE', 'GaussianCopula', 'WGAN_GP', 'CTABGAN']
Synthetic samples per generator: 1000


In [5]:
# ---------------------------------------------------
# SINGLE RUN - setup + CTABGAN
# ---------------------------------------------------
seed = SEED
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Train generators on 80% of the 1000-sample subset only.

# CTABGAN - train on raw categoricals (not one-hot encoded)
# Use country + page 1 + colour + location + model photography only.
# page 2 (clothing model) is too high-cardinality (217 products) for the train split.
CTABGAN_CATEGORICAL = [
    'country',
    'page 1 (main category)',
    'colour',
    'location',
    'model photography',
]
CTABGAN_MIXED = {}
CTABGAN_GENERAL = NUM_COLS + [target_col]
# Keep price as a continuous column in inverse_prep (do not force int rounding).
CTABGAN_INTEGER = []
CTABGAN_NUMERIC = NUM_COLS


def prepare_ctabgan_train_df(df, categorical_cols, numeric_cols, label_col):
    cols = categorical_cols + numeric_cols + [label_col]
    out = df[cols].copy()
    for col in categorical_cols:
        out[col] = out[col].fillna('None').astype(str)
    for col in numeric_cols + [label_col]:
        s = pd.to_numeric(out[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
        fill = s.median() if s.notna().any() else 0
        out[col] = s.fillna(fill).astype(np.float64)
    return out


def sanitize_ctabgan_sample(raw, data_prep, ref_df):
    """Replace NaN/inf in generator output before inverse_prep integer casting."""
    df = pd.DataFrame(raw, columns=data_prep.df.columns)
    rng = np.random.default_rng(SEED)

    for enc in getattr(data_prep, 'label_encoder_list', []):
        col = enc['column']
        n_classes = len(enc['label_encoder'].classes_)
        vals = pd.to_numeric(df[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
        vals = vals.fillna(0).clip(0, n_classes - 1)
        df[col] = np.round(vals)

    cols_to_fix = list(data_prep.integer_columns or []) + NUM_COLS
    cols_to_fix = list(dict.fromkeys(cols_to_fix))

    for col in cols_to_fix:
        if col not in df.columns or col not in ref_df.columns:
            continue
        ref = pd.to_numeric(ref_df[col], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
        if ref.empty:
            continue
        vals = pd.to_numeric(df[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
        if vals.notna().sum() == 0:
            df[col] = rng.choice(ref.values, size=len(df))
        else:
            fill = ref.median()
            vals = vals.fillna(fill).clip(ref.min(), ref.max())
            df[col] = vals

    return df.to_numpy()


def encode_ctabgan_synthetic(synth_df, reference_df, label_col):
    """Map CTABGAN raw output to the one-hot layout used by other generators."""
    return encode_ctabgan_to_onehot(
        synth_df, reference_df, train_real_ctab, label_col, seed=SEED
    )


if 'CTABGAN' in GENERATORS_TO_EVAL:
    import traceback
    try:
        data_path = 'online_shopping_train.csv'
        ctab_train = prepare_ctabgan_train_df(
            train_real_ctab, CTABGAN_CATEGORICAL, CTABGAN_NUMERIC, target_col
        )
        ctab_train.to_csv(data_path, index=False)
        ctabgan = CTABGAN(
            raw_csv_path=data_path,
            test_ratio=0.01,
            categorical_columns=CTABGAN_CATEGORICAL,
            log_columns=[],
            mixed_columns=CTABGAN_MIXED,
            general_columns=CTABGAN_GENERAL,
            integer_columns=CTABGAN_INTEGER,
            problem_type={'Regression': target_col}
        )
        ctabgan.synthesizer.epochs = CTABGAN_EPOCHS
        ctabgan.fit()
        raw_sample = ctabgan.synthesizer.sample(N_SYNTH_SAMPLES)
        raw_sample = sanitize_ctabgan_sample(raw_sample, ctabgan.data_prep, ctab_train)
        synthetic_ctabgan_raw = ctabgan.data_prep.inverse_prep(raw_sample)
        synthetic_ctabgan = encode_ctabgan_synthetic(
            synthetic_ctabgan_raw, train_real, target_col
        )
        synthetic_ctabgan = ensure_ml_ready_synthetic(synthetic_ctabgan, train_real, target_col)
        print(
            f'CTABGAN synth: unique rows={synthetic_ctabgan.drop(columns=[target_col]).drop_duplicates().shape[0]}, '
            f'price std={synthetic_ctabgan[target_col].std():.3f}'
        )
        synthetic_datasets['CTABGAN'] = synthetic_ctabgan.copy()
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_ctabgan,
                metadata=train_metadata
            )
            scores['CTABGAN'] = quality.get_score()
            print('CTABGAN:', round(scores['CTABGAN'], 4))
        else:
            print('CTABGAN: trained (quality eval skipped)')
    except Exception as e:
        print('CTABGAN Failed:', e)
        traceback.print_exc()
else:
    print('CTABGAN: skipped (not in GENERATORS_TO_EVAL)')


100%|██████████| 5/5 [00:26<00:00,  5.23s/it]


Finished training in 28.698818683624268  seconds.
CTABGAN synth: unique rows=1000, price std=3.113
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 291/291 [00:00<00:00, 1663.61it/s]|
Column Shapes Score: 99.65%

(2/2) Evaluating Column Pair Trends: |██████████| 42195/42195 [02:01<00:00, 346.98it/s]|
Column Pair Trends Score: 99.27%

Overall Score (Average): 99.46%

CTABGAN: 0.9946


In [6]:
# ---------------------------------------------------
# WGAN-GP
# ---------------------------------------------------
if 'WGAN_GP' in GENERATORS_TO_EVAL:
    import traceback

    WGAN_CONTINUOUS = NUM_COLS + [target_col]
    _wgan_all_cols = list(train_real.columns)
    _wgan_dummy_cols = [c for c in _wgan_all_cols if c not in WGAN_CONTINUOUS]
    _wgan_dummy_idx = [_wgan_all_cols.index(c) for c in _wgan_dummy_cols]

    def postprocess_wgan_synthetic(synth_df, reference_df, continuous_cols):
        out = synth_df.reindex(columns=reference_df.columns).copy()
        dummy_cols = [c for c in reference_df.columns if c not in continuous_cols]
        for col in dummy_cols:
            # Generator already applies sigmoid; keep in (0, 1).
            out[col] = out[col].clip(0, 1)
        for col in continuous_cols:
            if col in out.columns:
                out[col] = out[col].clip(reference_df[col].min(), reference_df[col].max())
        return out.apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float64)

    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        data_wgan = train_real.copy()
        data_wgan = data_wgan.apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float32)
        real_tensor = torch.tensor(data_wgan.to_numpy(), dtype=torch.float32)

        batch_size = min(32, len(train_real))
        latent_dim = 64
        data_dim = real_tensor.shape[1]
        loader = torch.utils.data.DataLoader(real_tensor, batch_size=batch_size, shuffle=True, drop_last=False)

        class Generator(nn.Module):
            def __init__(self, dummy_indices):
                super().__init__()
                self.dummy_indices = dummy_indices
                self.model = nn.Sequential(
                    nn.Linear(latent_dim, 128),
                    nn.LayerNorm(128),
                    nn.LeakyReLU(0.2),
                    nn.Linear(128, 256),
                    nn.LayerNorm(256),
                    nn.LeakyReLU(0.2),
                    nn.Linear(256, data_dim),
                )

            def forward(self, z):
                raw = self.model(z)
                if self.dummy_indices:
                    raw = raw.clone()
                    raw[:, self.dummy_indices] = torch.sigmoid(raw[:, self.dummy_indices])
                return raw

        class Critic(nn.Module):
            def __init__(self):
                super().__init__()
                self.model = nn.Sequential(
                    nn.Linear(data_dim, 256),
                    nn.LeakyReLU(0.2),
                    nn.Linear(256, 128),
                    nn.LeakyReLU(0.2),
                    nn.Linear(128, 1),
                )
            def forward(self, x):
                return self.model(x)

        generator = Generator(_wgan_dummy_idx).to(device)
        critic = Critic().to(device)
        optimizer_G = optim.Adam(generator.parameters(), lr=0.0001, betas=(0.5, 0.9))
        optimizer_C = optim.Adam(critic.parameters(), lr=0.0001, betas=(0.5, 0.9))

        def gradient_penalty(critic_model, real_samples, fake_samples):
            alpha = torch.rand(real_samples.size(0), 1, device=device).expand_as(real_samples)
            interpolates = (alpha * real_samples + (1 - alpha) * fake_samples).requires_grad_(True)
            critic_interpolates = critic_model(interpolates)
            gradients = torch.autograd.grad(
                outputs=critic_interpolates,
                inputs=interpolates,
                grad_outputs=torch.ones_like(critic_interpolates),
                create_graph=True,
                retain_graph=True
            )[0]
            gradients = gradients.view(gradients.size(0), -1)
            return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

        for _ in range(WGAN_EPOCHS):
            for real_batch in loader:
                real_batch = real_batch.to(device)
                for _ in range(5):
                    z = torch.randn(real_batch.size(0), latent_dim, device=device)
                    fake_batch = generator(z).detach()
                    critic_real = critic(real_batch).mean()
                    critic_fake = critic(fake_batch).mean()
                    gp = gradient_penalty(critic, real_batch, fake_batch)
                    critic_loss = critic_fake - critic_real + 10 * gp
                    optimizer_C.zero_grad()
                    critic_loss.backward()
                    optimizer_C.step()
                z = torch.randn(real_batch.size(0), latent_dim, device=device)
                fake = generator(z)
                generator_loss = -critic(fake).mean()
                optimizer_G.zero_grad()
                generator_loss.backward()
                optimizer_G.step()

        generator.eval()
        synth_chunks = []
        sample_batch = 256
        with torch.no_grad():
            for start in range(0, N_SYNTH_SAMPLES, sample_batch):
                n = min(sample_batch, N_SYNTH_SAMPLES - start)
                z = torch.randn(n, latent_dim, device=device)
                synth_chunks.append(generator(z).cpu().numpy())
        synthetic_scaled = np.vstack(synth_chunks)
        synthetic_wgan = pd.DataFrame(synthetic_scaled, columns=data_wgan.columns)
        synthetic_wgan = postprocess_wgan_synthetic(synthetic_wgan, train_real, WGAN_CONTINUOUS)
        synthetic_wgan = ensure_ml_ready_synthetic(synthetic_wgan, train_real, target_col)
        print(
            f'WGAN_GP synth: unique rows={synthetic_wgan.drop(columns=[target_col]).drop_duplicates().shape[0]}, '
            f'price std={synthetic_wgan[target_col].std():.3f}'
        )
        synthetic_datasets['WGAN_GP'] = synthetic_wgan.copy()
        print('WGAN_GP: synthesis complete')

        del generator, critic, real_tensor
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        print('WGAN_GP Failed (training/sampling):')
        traceback.print_exc()

    if 'WGAN_GP' in synthetic_datasets and RUN_QUALITY_EVAL:
        try:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_datasets['WGAN_GP'],
                metadata=train_metadata
            )
            scores['WGAN_GP'] = quality.get_score()
            print('WGAN_GP:', round(scores['WGAN_GP'], 4))
        except Exception as e:
            print('WGAN_GP quality eval failed:', e)
            traceback.print_exc()
else:
    print('WGAN_GP: skipped (not in GENERATORS_TO_EVAL)')

WGAN_GP synth: unique rows=1000, price std=1.773
WGAN_GP: synthesis complete
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 291/291 [00:03<00:00, 79.44it/s]|
Column Shapes Score: 2.04%

(2/2) Evaluating Column Pair Trends: |██████████| 42195/42195 [02:59<00:00, 235.58it/s]|
Column Pair Trends Score: 0.04%

Overall Score (Average): 1.04%

WGAN_GP: 0.0104


In [7]:
# ---------------------------------------------------
# SDV Models (CTGAN, CopulaGAN, TVAE, GaussianCopula)
# ---------------------------------------------------
sdv_models = {
    'CTGAN': CTGANSynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    'CopulaGAN': CopulaGANSynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    'TVAE': TVAESynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    'GaussianCopula': GaussianCopulaSynthesizer(metadata=train_metadata),
}

for model_name, model in sdv_models.items():
    if model_name not in GENERATORS_TO_EVAL:
        print(f'{model_name}: skipped (not in GENERATORS_TO_EVAL)')
        continue
    try:
        model.fit(train_real)
        synthetic_data = model.sample(N_SYNTH_SAMPLES)
        synthetic_data = align_to_train_schema(synthetic_data, train_real, target_col)
        synthetic_datasets[model_name] = synthetic_data.copy()
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_data,
                metadata=train_metadata
            )
            scores[model_name] = quality.get_score()
            print(f'{model_name}: {round(scores[model_name], 4)}')
        else:
            print(f'{model_name}: trained (quality eval skipped)')
    except Exception as e:
        print(f'{model_name} Failed: {e}')

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 291/291 [00:00<00:00, 821.71it/s]|
Column Shapes Score: 69.16%

(2/2) Evaluating Column Pair Trends: |██████████| 42195/42195 [02:11<00:00, 320.41it/s]|
Column Pair Trends Score: 48.75%

Overall Score (Average): 58.96%

CTGAN: 0.5896
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 291/291 [00:00<00:00, 813.10it/s]|
Column Shapes Score: 69.44%

(2/2) Evaluating Column Pair Trends: |██████████| 42195/42195 [02:12<00:00, 318.48it/s]|
Column Pair Trends Score: 49.12%

Overall Score (Average): 59.28%

CopulaGAN: 0.5928
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 291/291 [00:00<00:00, 1950.21it/s]|
Column Shapes Score: 96.99%

(2/2) Evaluating Column Pair Trends: |██████████| 42195/42195 [02:14<00:00, 314.46it/s]|
Column Pair Trends Score: 93.65%

Overall Score (Average): 95.32%

TVAE: 0.9532
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 291/291 [00:00<00:00, 

In [8]:
# Quality summary
quality_results = []
for gen_name in GENERATORS_TO_EVAL:
    quality_results.append({
        'Generator': gen_name,
        'Quality_Score': scores.get(gen_name, np.nan),
        'Status': 'Success' if gen_name in synthetic_datasets else 'Failed'
    })

quality_df = pd.DataFrame(quality_results).sort_values('Quality_Score', ascending=False)
display(quality_df)

,Generator,Quality_Score,Status
3,GaussianCopula,0.994904,Success
5,CTABGAN,0.994625,Success
2,TVAE,0.953211,Success
1,CopulaGAN,0.592804,Success
0,CTGAN,0.589575,Success
4,WGAN_GP,0.010415,Success


In [9]:
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor

regressors = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.001, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000),
    'SVR_RBF': SVR(kernel='rbf', C=1.0, epsilon=0.1),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoost': GradientBoostingRegressor(random_state=42),
}

print(f'Regression evaluation: {len(regressors)} models, {len(EVAL_SEEDS)} seeds, {len(GENERATORS_TO_EVAL)} generators')

Regression evaluation: 10 models, 10 seeds, 6 generators


In [10]:
def evaluate_regression_models(train_df, test_df, label_col, models, test_size=0.2, seeds=None, use_holdout=False, schema_df=None):
    if seeds is None:
        seeds = EVAL_SEEDS
    if schema_df is None:
        schema_df = test_df if use_holdout else train_df

    def _needs_align(df, reference_df):
        if list(df.columns) != list(reference_df.columns):
            return True
        return df.select_dtypes(include=['object', 'string', 'category']).shape[1] > 0

    if _needs_align(train_df, schema_df):
        train_df = align_to_train_schema(train_df, schema_df, label_col)
    if _needs_align(test_df, schema_df):
        test_df = align_to_train_schema(test_df, schema_df, label_col)

    feature_cols = [c for c in train_df.columns if c != label_col]
    X_full_ref = train_df[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
    keep_features = X_full_ref.columns[X_full_ref.var() > 1e-10].tolist()
    if not keep_features:
        keep_features = feature_cols

    results = []

    def _std(values):
        return float(np.std(values, ddof=1)) if len(values) > 1 else 0.0

    for name, model in models.items():
        r2_scores = []
        mse_scores = []
        rmse_scores = []
        mae_scores = []

        for seed in seeds:
            X_full = train_df[keep_features]
            y_full = train_df[label_col]
            X_test = test_df[keep_features]
            y_test = test_df[label_col]

            if use_holdout:
                n_train = max(2, int(len(X_full) * (1 - test_size)))
                rng = np.random.default_rng(seed)
                idx = rng.choice(len(X_full), size=n_train, replace=True)
                X_train = X_full.iloc[idx].reset_index(drop=True)
                y_train = y_full.iloc[idx].reset_index(drop=True)
            else:
                X_train, _, y_train, _ = train_test_split(
                    X_full, y_full, test_size=test_size, random_state=seed
                )
                _, X_test, _, y_test = train_test_split(
                    X_test, y_test, test_size=test_size, random_state=seed
                )

            reg = clone(model)
            if 'random_state' in reg.get_params():
                reg.set_params(random_state=seed)

            reg.fit(X_train, y_train)
            y_pred = reg.predict(X_test)

            mse = mean_squared_error(y_test, y_pred)
            r2_scores.append(r2_score(y_test, y_pred))
            mse_scores.append(mse)
            rmse_scores.append(np.sqrt(mse))
            mae_scores.append(mean_absolute_error(y_test, y_pred))

        results.append({
            'Model': name,
            'R2 Mean': np.mean(r2_scores),
            'R2 Std': _std(r2_scores),
            'MSE Mean': np.mean(mse_scores),
            'MSE Std': _std(mse_scores),
            'RMSE Mean': np.mean(rmse_scores),
            'RMSE Std': _std(rmse_scores),
            'MAE Mean': np.mean(mae_scores),
            'MAE Std': _std(mae_scores),
            'R2 (Mean ± SD)': f"{np.mean(r2_scores):.4f} ± {_std(r2_scores):.4f}",
            'MSE (Mean ± SD)': f"{np.mean(mse_scores):.4f} ± {_std(mse_scores):.4f}",
            'RMSE (Mean ± SD)': f"{np.mean(rmse_scores):.4f} ± {_std(rmse_scores):.4f}",
            'MAE (Mean ± SD)': f"{np.mean(mae_scores):.4f} ± {_std(mae_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by='R2 Mean', ascending=False)


In [11]:
print('TRTR (Train Real, Test Real) - 80% train / 20% holdout')
trtr_results = evaluate_regression_models(
    train_df=train_real,
    test_df=test_real,
    label_col=target_col,
    models=regressors,
    seeds=EVAL_SEEDS,
    use_holdout=True,
    schema_df=train_real,
)
display(trtr_results[['Model', 'R2 (Mean ± SD)', 'MSE (Mean ± SD)', 'RMSE (Mean ± SD)', 'MAE (Mean ± SD)']])

all_comparisons = []
for synth_name in GENERATORS_TO_EVAL:
    if synth_name not in synthetic_datasets:
        print(f'Skipping {synth_name} - no synthetic dataset')
        continue

    print(f'{synth_name} - TSTR (train on synthetic, test on 20% holdout)')
    tstr_results = evaluate_regression_models(
        train_df=synthetic_datasets[synth_name],
        test_df=test_real,
        label_col=target_col,
        models=regressors,
        seeds=EVAL_SEEDS,
        use_holdout=True,
        schema_df=train_real,
    )
    display(tstr_results[['Model', 'R2 (Mean ± SD)', 'MSE (Mean ± SD)', 'RMSE (Mean ± SD)', 'MAE (Mean ± SD)']])

    comparison = trtr_results.merge(tstr_results, on='Model', suffixes=('_TRTR', '_TSTR'))
    comparison['R2_Drop'] = comparison['R2 Mean_TRTR'] - comparison['R2 Mean_TSTR']
    comparison['MSE_Increase'] = comparison['MSE Mean_TSTR'] - comparison['MSE Mean_TRTR']
    comparison['RMSE_Increase'] = comparison['RMSE Mean_TSTR'] - comparison['RMSE Mean_TRTR']
    comparison['MAE_Increase'] = comparison['MAE Mean_TSTR'] - comparison['MAE Mean_TRTR']
    comparison['Synthetic_Model'] = synth_name
    all_comparisons.append(comparison)

combined_comparison = pd.concat(all_comparisons, ignore_index=True)
summary = (
    combined_comparison
    .groupby('Synthetic_Model', as_index=False)[['R2_Drop', 'MSE_Increase', 'RMSE_Increase', 'MAE_Increase']]
    .mean()
    .sort_values('R2_Drop')
)

display(summary)


TRTR (Train Real, Test Real) - 80% train / 20% holdout


,Model,R2 (Mean ± SD),MSE (Mean ± SD),RMSE (Mean ± SD),MAE (Mean ± SD)
0,LinearRegression,0.9670 ± 0.0099,6.0008 ± 1.7998,2.4226 ± 0.3827,0.6783 ± 0.1185
2,Lasso,0.9662 ± 0.0105,6.1598 ± 1.9156,2.4531 ± 0.3976,0.8192 ± 0.1148
3,ElasticNet,0.9647 ± 0.0097,6.4218 ± 1.7673,2.5107 ± 0.3625,1.0648 ± 0.0960
8,ExtraTrees,0.9616 ± 0.0119,6.9949 ± 2.1666,2.6126 ± 0.4334,0.7496 ± 0.1430
7,RandomForest,0.9601 ± 0.0095,7.2606 ± 1.7212,2.6764 ± 0.3288,1.1706 ± 0.0908
1,Ridge,0.9576 ± 0.0090,7.7189 ± 1.6444,2.7633 ± 0.3043,1.4837 ± 0.0875
6,DecisionTree,0.9509 ± 0.0182,8.9375 ± 3.3182,2.9440 ± 0.5478,0.9005 ± 0.2423
9,GradientBoost,0.9374 ± 0.0102,11.3970 ± 1.8639,3.3655 ± 0.2792,2.5092 ± 0.0973
5,KNN,0.0777 ± 0.0680,167.8871 ± 12.3865,12.9491 ± 0.4809,10.2263 ± 0.3617
4,SVR_RBF,0.0024 ± 0.0236,181.5960 ± 4.2918,13.4749 ± 0.1571,10.5663 ± 0.0347


CTGAN - TSTR (train on synthetic, test on 20% holdout)


,Model,R2 (Mean ± SD),MSE (Mean ± SD),RMSE (Mean ± SD),MAE (Mean ± SD)
4,SVR_RBF,-0.0206 ± 0.0085,185.7890 ± 1.5424,13.6303 ± 0.0565,10.9247 ± 0.0412
7,RandomForest,-0.0220 ± 0.0256,186.0438 ± 4.6598,13.6388 ± 0.1707,11.0792 ± 0.2178
9,GradientBoost,-0.0257 ± 0.0360,186.7186 ± 6.5572,13.6626 ± 0.2389,11.1480 ± 0.3126
1,Ridge,-0.2854 ± 0.3395,233.9861 ± 61.7995,15.1914 ± 1.8875,12.6111 ± 1.7932
3,ElasticNet,-0.2892 ± 0.3436,234.6843 ± 62.5477,15.2123 ± 1.9062,12.6308 ± 1.8077
2,Lasso,-0.2912 ± 0.3454,235.0338 ± 62.8756,15.2229 ± 1.9143,12.6406 ± 1.8138
0,LinearRegression,-0.2957 ± 0.3517,235.8559 ± 64.0228,15.2465 ± 1.9433,12.6640 ± 1.8372
5,KNN,-0.3630 ± 0.1019,248.1165 ± 18.5477,15.7419 ± 0.5871,12.6349 ± 0.4982
6,DecisionTree,-0.3976 ± 0.3358,254.4090 ± 61.1197,15.8594 ± 1.7912,12.4770 ± 1.3499
8,ExtraTrees,-0.6163 ± 0.7027,294.2175 ± 127.9122,16.8391 ± 3.4422,13.4231 ± 3.2618


CopulaGAN - TSTR (train on synthetic, test on 20% holdout)


...

WGAN_GP - TSTR (train on synthetic, test on 20% holdout)


,Model,R2 (Mean ± SD),MSE (Mean ± SD),RMSE (Mean ± SD),MAE (Mean ± SD)
9,GradientBoost,-0.5110 ± 0.2228,275.0489 ± 40.5601,16.5448 ± 1.2104,12.6139 ± 1.0213
7,RandomForest,-2.1991 ± 0.2241,582.3544 ± 40.8020,24.1181 ± 0.8642,20.1977 ± 0.9697
6,DecisionTree,-2.3249 ± 0.7792,605.2511 ± 141.8466,24.4503 ± 2.8742,20.6082 ± 3.1607
8,ExtraTrees,-2.3586 ± 0.0429,611.3787 ± 7.8108,24.7256 ± 0.1581,20.8755 ± 0.1737
5,KNN,-3.2118 ± 0.0193,766.6984 ± 3.5109,27.6893 ± 0.0634,24.1520 ± 0.0632
1,Ridge,-3.3655 ± 0.1358,794.6655 ± 24.7242,28.1867 ± 0.4388,24.7215 ± 0.5352
3,ElasticNet,-3.3666 ± 0.1242,794.8732 ± 22.6066,28.1910 ± 0.3983,24.7224 ± 0.4808
4,SVR_RBF,-3.4624 ± 0.0185,812.3077 ± 3.3686,28.5010 ± 0.0592,25.0966 ± 0.0667
2,Lasso,-3.5565 ± 0.1492,829.4312 ± 27.1590,28.7963 ± 0.4738,25.3667 ± 0.6248
0,LinearRegression,-698.6802 ± 265.7744,127365.8651 ± 48380.0880,351.1880 ± 66.9400,274.7237 ± 51.3119


CTABGAN - TSTR (train on synthetic, test on 20% holdout)


,Model,R2 (Mean ± SD),MSE (Mean ± SD),RMSE (Mean ± SD),MAE (Mean ± SD)
4,SVR_RBF,-0.0315 ± 0.0041,187.7743 ± 0.7420,13.7031 ± 0.0271,10.8720 ± 0.0166
9,GradientBoost,-0.0380 ± 0.0248,188.9549 ± 4.5107,13.7452 ± 0.1627,10.9246 ± 0.1008
7,RandomForest,-0.0448 ± 0.0255,190.1926 ± 4.6435,13.7901 ± 0.1675,10.9497 ± 0.1083
8,ExtraTrees,-0.0578 ± 0.0502,192.5480 ± 9.1425,13.8727 ± 0.3254,11.0535 ± 0.2494
1,Ridge,-0.0620 ± 0.0194,193.3293 ± 3.5260,13.9038 ± 0.1270,11.1654 ± 0.1214
3,ElasticNet,-0.0649 ± 0.0222,193.8441 ± 4.0401,13.9221 ± 0.1449,11.1849 ± 0.1237
2,Lasso,-0.0678 ± 0.0259,194.3680 ± 4.7099,13.9407 ± 0.1681,11.1996 ± 0.1266
5,KNN,-0.0680 ± 0.0123,194.4040 ± 2.2412,13.9427 ± 0.0804,11.1277 ± 0.0828
0,LinearRegression,-0.0731 ± 0.0274,195.3373 ± 4.9862,13.9753 ± 0.1777,11.2444 ± 0.1439
6,DecisionTree,-0.0962 ± 0.0480,199.5444 ± 8.7376,14.1230 ± 0.3085,11.2036 ± 0.2167


,Synthetic_Model,R2_Drop,MSE_Increase,RMSE_Increase,MAE_Increase
3,GaussianCopula,0.455725,82.957684,6.128955,5.473560
0,CTABGAN,0.834964,151.992243,9.074635,8.075695
1,CTGAN,1.035233,188.448008,10.207300,9.206509
4,TVAE,1.253429,228.167263,11.506650,10.217182
2,CopulaGAN,1.354257,246.521316,11.799224,10.239107
5,WGAN_GP,73.078220,13302.749999,53.421874,44.290981


In [12]:
output_file = 'TRTR_TSTR_results_regression.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    quality_df.to_excel(writer, sheet_name='Quality_Metrics', index=False)
    trtr_results.to_excel(writer, sheet_name='TRTR_Results', index=False)
    combined_comparison.to_excel(writer, sheet_name='All_Comparisons', index=False)
    summary.to_excel(writer, sheet_name='Summary', index=False)
    for synth_name in GENERATORS_TO_EVAL:
        if synth_name in combined_comparison['Synthetic_Model'].values:
            synth_results = combined_comparison[combined_comparison['Synthetic_Model'] == synth_name]
            synth_results.to_excel(writer, sheet_name=synth_name[:31], index=False)

print(f'Results saved to: {output_file}')

Results saved to: TRTR_TSTR_results_regression.xlsx
